In [19]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, datetime, timezone
import time
import re
import sys
import subprocess
import jinja2
import matplotlib.pyplot as plt

In [20]:
# Configuration
today = date.today()
now = datetime.now(timezone.utc)
year_start = 2022
file_name_data = Path('data/assets-class-data.csv')
file_name_source = Path('data/assets2.csv')
use_cached_data = False
report_output_dir = Path('output')
# output_dir.mkdir(exist_ok=True)

# Data
returns_data = pd.read_csv(file_name_source)
tickers = returns_data['Ticker'].tolist()

In [21]:
# returns_data

In [22]:
# Download the historical data for the tickers
def modified_within_last_minute(path):
    if not path.exists():
        return False
    mtime = datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc)
    return (now - mtime).total_seconds() <= 300  # 300 seconds = 5 minutes

source_just_changed = modified_within_last_minute(file_name_source)
cache_missing       = not file_name_data.exists()

# if (file_name.exists() and pd.to_datetime(file_name.stat().st_mtime, unit='s').date() == today and file_name_source.exists()) or (pd.to_datetime(file_name_source.stat().st_mtime, unit='s').date() == today):
# if cache_missing or source_just_changed:
if use_cached_data:
    assets_class_data = pd.read_csv(file_name_data, header=[0, 1], index_col=0, parse_dates=True)
else:
    assets_class_data = yf.download(tickers, interval='1mo', start='2020-01-01', end=today, progress=False)
    assets_class_data.to_csv(file_name_data)

/tmp/ipykernel_28032/2169142778.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  assets_class_data = yf.download(tickers, interval='1mo', start='2020-01-01', end=today, progress=False)


In [23]:
# Append quarter columns for 2020-2027 with "Close" values
current_year = datetime.now().year
current_quarter = (datetime.now().month - 1) // 3 + 1

for year in range(year_start, current_year + 1):
    max_quarter = current_quarter if year == current_year else 4

    for quarter in range(1, max_quarter + 1):
        returns_data[f"{year}-Q{quarter}"] = np.nan

# Fill the quarter columns with the "Close" values from the assets_class_data DataFrame
assets_class_data = assets_class_data.sort_index()
for ticker in tickers:
    for year in range(year_start, current_year + 1):
        max_quarter = current_quarter if year == current_year else 4

        for quarter in range(1, max_quarter + 1):
            quarter_end_month = quarter * 3
            quarter_end_date = pd.Timestamp(year=year, month=quarter_end_month, day=1)

            available = assets_class_data.loc[:quarter_end_date]
            if not available.empty:
                last_date = available.index.max()
                price = available.loc[last_date, ('Close', ticker)]
                returns_data.loc[returns_data['Ticker'] == ticker, f"{year}-Q{quarter}"] = round(price, 2)


In [24]:
returns_data


,Ticker,Sector,Class,Name,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,...,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1,2026-Q2
0,DBC,Commodities,Commodities,Invesco DB Commodity Index Tracking Fund,22.68,23.18,20.81,21.45,20.78,19.87,...,21.10,21.33,20.45,19.64,21.77,21.10,21.80,21.63,28.95,27.63
1,QQQ,Technology,US Large Cap Growth,Invesco QQQ Trust,353.07,273.30,261.10,260.61,314.85,362.98,...,438.61,473.91,483.54,507.19,465.97,548.98,598.15,612.75,576.45,740.62
2,IWM,Equity,US Small Cap,iShares Russell 2000 ETF,194.06,160.42,156.63,166.45,171.09,180.26,...,204.69,197.98,216.15,216.95,196.45,212.98,239.47,244.32,246.97,295.59
3,VWO,International,Emerging Markets,Vanguard FTSE Emerging Markets ETF,40.23,36.43,32.15,34.81,36.67,36.95,...,39.29,41.20,45.22,41.74,43.93,48.06,52.80,52.66,53.99,60.77
4,MDY,Equity,US Mid Cap,SPDR S&P MidCap 400 ETF Trust,463.87,392.00,382.12,423.00,439.56,460.46,...,540.45,520.73,556.46,557.99,524.54,558.34,589.38,598.15,613.72,691.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,IYH,Healthcare,Stock,iShares U.S. Healthcare ETF (Healthcare),54.73,51.09,48.17,54.23,52.34,53.88,...,60.06,59.61,63.37,57.01,59.78,55.60,57.97,64.49,61.29,62.68
124,AIOZ-USD,Cryptocurrency,Cryptocurrency,AIOZ Network USD,0.24,0.05,0.05,0.03,0.03,0.01,...,0.94,0.55,0.50,0.79,0.24,0.31,0.27,0.09,0.06,0.06
125,NEAR-USD,Cryptocurrency,Cryptocurrency,NEAR Protocol USD,13.27,3.33,3.56,1.26,1.99,1.38,...,7.30,5.30,5.29,4.90,2.51,2.15,2.63,1.51,1.19,2.24
126,AVAX-USD,Cryptocurrency,Cryptocurrency,AVAX USD,96.92,16.93,17.20,10.90,17.72,13.01,...,54.11,29.36,27.74,35.69,18.77,17.97,30.00,12.30,8.91,6.31


In [25]:
# The issue with teh previous approach is that pct_change() is computing the change down the rows (across different tickers), but we want it computed across
# columns (across quarters) for each row. pct_change() defaults to axis=0 (row-wise, comparing ticker to ticker) — we needed axis=1 (column-wise, comparing
#  quarter to quarter). The first quarter column (e.g. 2022-Q1) will always be NaN after the change since there's no prior period, so it gets dropped rather
#  than using dropna() which would drop entire rows
# The regex filter isolates only quarter columns, leaving Ticket and Asset_Class untouched
# For the example row, 2026-Q2 would correctly show (30.70 - 28.95) / 28.95 * 100 = +6.04%.

returns_pctchange = returns_data.copy()

# Identify quarter columns
quarter_cols = [col for col in returns_data.columns if re.match(r'\d{4}-Q\d', col)]

# Calculate pct change across columns (axis=1), then overwrite those columns
pct_values = returns_data[quarter_cols].pct_change(axis=1) * 100

for col in quarter_cols:
    returns_pctchange[col] = pct_values[col].round(1)

# Drop the first quarter column since it has no prior period to compare against
# returns_pctchange.drop(columns=[quarter_cols[0]], inplace=True)

/tmp/ipykernel_28032/2794772260.py:14: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pct_values = returns_data[quarter_cols].pct_change(axis=1) * 100


In [26]:
# Append two additional, new change columns: Cumulative and CAGR(Annualized)
# Cumulative Change is straightforward — it's just the total return from the first period's value to the last, regardless of how many quarters are in between.
# CAGR derives exact years from the column names themselves (e.g. 2022-Q1 to 2026-Q2 = ~4.25 years) rather than just counting quarters, which gives a more
# precise annualization. For your example row: (30.70 / 22.68)^(1/4.25) - 1 ≈ +7.7% annualized.
# Both columns operate on the original returns_data values (not the pct-change values), which is the correct basis for these calculations.

# Drop the first quarter col (no prior period)
first_col = quarter_cols[0]
returns_pctchange.drop(columns=[first_col], inplace=True)

# Remaining quarter cols after dropping the first
remaining_quarter_cols = [col for col in quarter_cols if col != first_col]

# --- Cumulative Change ---
# ( final_value / first_value - 1 ) * 100
first_val = returns_data[first_col]
last_val  = returns_data[quarter_cols[-1]]
returns_pctchange["Cumulative"] = ((last_val / first_val - 1) * 100).round(1)

# --- CAGR (Annualized) ---
# Derive start/end dates from column names to get exact number of years
def col_to_date(col):
    year, q = col.split("-Q")
    month = int(q) * 3          # end-of-quarter month: Q1→3, Q2→6, Q3→9, Q4→12
    return pd.Timestamp(year=int(year), month=month, day=1) + pd.offsets.MonthEnd(0)

start_date = col_to_date(first_col)
end_date   = col_to_date(quarter_cols[-1])
years      = (end_date - start_date).days / 365.25

returns_pctchange["CAGR"] = (
    ((last_val / first_val) ** (1 / years) - 1) * 100
).round(2)

returns_pctchange.sort_values(by=["Sector", "2026-Q2"], ascending=[True, False], inplace=True)

In [27]:

# print(sys.executable)
# Force install matplotlib into the EXACT runtime currently executing this code
# subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])

# 1. Identify your text columns vs your numeric columns
text_cols = ['Ticker','Sector', 'Class', 'Name']
numeric_cols = [col for col in returns_pctchange.columns if col not in text_cols]

# 2. Chain styles, restricting formatting and colors ONLY to the numeric columns
styled_df = (
    returns_pctchange
    .drop(columns=["Class","Name"], inplace=False)
    .style
    # Chain multiple styles together
    .format("{:.1f}", subset=numeric_cols, na_rep="-")  # Format only numbers.format("{:.1f}", na_rep="-")  # Displays 1 decimal place; turns missing values into "-"
    # .background_gradient(cmap="Greens")
    # .highlight_max(color="green")
    # .highlight_min(color="red")
    .background_gradient(
        cmap="RdYlGn",      # Red → Yellow → Green (diverging)
        subset=numeric_cols,
        vmin=-10,           # Adjust to your data's typical min
        vmax=30             # Adjust to your data's typical max
    )
)

# Export the styled DataFrame to an HTML file.
filename = (
    report_output_dir
    / f"market_years_report_{datetime.now().strftime('%Y%m%d')}.html"
)
styled_df.to_html(filename)

# print(returns_pctchange.dtypes)
styled_df


,Ticker,Sector,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4,2026-Q1,2026-Q2,Cumulative,CAGR
38,CPER,Commodities,-22.6,-8.0,12.9,8.0,-7.0,-0.8,4.9,4.1,8.3,4.2,-11.3,25.6,0.1,-5.2,16.5,-1.5,12.9,35.4,7.4
39,COPX,Commodities,-33.3,-4.9,25.3,10.5,-2.7,-2.0,3.0,14.3,6.3,5.2,-19.2,3.7,15.2,33.6,20.0,8.8,12.0,108.1,18.8
61,URA,Commodities,-29.0,7.4,1.3,-0.5,8.9,24.6,2.4,10.4,0.4,-1.1,-6.4,-12.1,69.3,22.8,-10.4,18.8,-1.3,110.3,19.1
0,DBC,Commodities,2.2,-10.2,3.1,-3.1,-4.4,10.0,-11.7,9.4,1.1,-4.1,-4.0,10.8,-3.1,3.3,-0.8,33.8,-4.6,21.8,4.8
34,URNM,Commodities,-31.3,13.7,-2.3,-1.1,7.0,40.5,2.3,5.9,-0.1,-6.8,-12.1,-17.2,47.7,26.1,-9.2,18.9,-9.0,52.4,10.4
7,GLD,Commodities,-6.7,-8.2,9.7,8.0,-2.7,-3.8,11.5,7.6,4.5,13.0,-0.4,19.0,5.8,16.6,11.5,8.6,-10.0,114.3,19.6
37,SLV,Commodities,-18.5,-6.1,25.8,0.5,-5.6,-2.6,7.1,4.5,16.8,6.9,-7.3,17.7,5.9,29.1,52.0,5.8,-12.7,160.1,25.2
85,DIS,Communication Services,-31.2,-0.1,-7.9,15.3,-10.8,-9.2,11.4,36.0,-18.9,-2.7,15.8,-11.0,25.6,-7.3,-0.6,-14.7,7.8,-22.5,-5.8
83,HD,Consumer Discretionary,-7.8,1.9,15.2,-6.6,6.8,-2.1,15.5,10.7,-9.1,18.4,-3.5,-5.8,0.7,11.2,-14.6,-3.8,3.1,25.3,5.5
84,MCD,Consumer Discretionary,-0.2,-5.5,14.9,6.7,6.7,-10.8,13.2,-4.4,-9.6,20.3,-4.2,8.4,-5.9,4.6,1.1,2.3,-9.2,24.3,5.3
